In [1]:
# 1) Quick health check
import requests, numpy as np

# pick ONE of these:
# BASE = "http://bge-m3:8000"                  # same compose network
BASE = "http://host.docker.internal:8800"  # via host port mapping (e.g., 8800:8000)

r = requests.get(f"{BASE}/health", timeout=20)
r.raise_for_status()
r.json()


{'ok': True,
 'device': 'NVIDIA GeForce RTX 4090',
 'cuda': True,
 'model': 'BAAI/bge-m3'}

In [2]:
# 2) Single encode (dense)

text = "Hormigón armado HA-25 en zapata corrida, encofrado y vibrado."
payload = {"text": text, "mode": "dense"}  # or add "max_length": 256

r = requests.post(f"{BASE}/encode", json=payload, timeout=300)
r.raise_for_status()
vec = np.array(r.json()["vec"], dtype="float32")
vec.shape, vec[:8]  # look at length & first values


((1024,),
 array([-0.00193167,  0.03264586, -0.01557543,  0.04420921, -0.0076733 ,
        -0.04213452,  0.01189133,  0.06559683], dtype=float32))

In [3]:
# 3) Batch encode (dense)

texts = [
    "Excavación en zanja, terreno de consistencia media.",
    "Relleno y compactación con zahorra artificial.",
    "Hormigón armado HA-25 en zapata corrida, encofrado y vibrado."
]
payload = {"texts": texts, "mode": "dense"}  # or "max_length": 256/512

r = requests.post(f"{BASE}/encode_batch", json=payload, timeout=120)
r.raise_for_status()
vecs = r.json()["vecs"]               # list[list[float]]
import numpy as np
X = np.array(vecs, dtype="float32")   # shape: (B, 1024)
X.shape


(3, 1024)

In [4]:
# 4) Sparse encode

payload = {"text": "hormigón armado HA-25", "mode": "sparse"}  # or batch with /encode_batch
r = requests.post(f"{BASE}/encode", json=payload, timeout=60)
r.raise_for_status()
lw = r.json()["lexical_weights"]      # dict: token_id(str) -> weight(float)
list(lw.items())[:10]


[('3328', 0.07647705078125),
 ('58849', 0.1728515625),
 ('5800', 0.1041259765625),
 ('189938', 0.1944580078125),
 ('11666', 0.2181396484375),
 ('28174', 0.295166015625)]

In [5]:
import requests, json, time, math, random
import numpy as np

BASE = "http://host.docker.internal:8800"  # no trailing slash

def show(resp, label):
    try:
        js = resp.json()
        print(f"[{label}] {resp.status_code} | keys: {list(js.keys())}")
        return js
    except Exception as e:
        print(f"[{label}] parse error:", e, "| text:", resp.text[:500])
        return None

def l2norm_ok(vec, tol=5e-3):
    v = np.asarray(vec, dtype=np.float32)
    n = float(np.linalg.norm(v) + 1e-12)
    return abs(n - 1.0) < tol

def l2norm_tokens_ok(mats, tol=5e-3, sample=3):
    """Check a few random token rows for L2≈1"""
    if not mats: return True
    rng = random.Random(0)
    for _ in range(min(sample, len(mats))):
        M = np.asarray(mats[rng.randrange(len(mats))], dtype=np.float32)
        if M.size == 0: 
            continue
        ridx = rng.randrange(M.shape[0])
        n = float(np.linalg.norm(M[ridx]) + 1e-12)
        if abs(n - 1.0) >= tol:
            return False
    return True

def time_call(fn, *a, repeat=1, label="call", **kw):
    t0 = time.perf_counter()
    out = None
    for _ in range(repeat):
        out = fn(*a, **kw)
    dt = (time.perf_counter() - t0) / repeat
    print(f"[time] {label}: {dt*1000:.1f} ms/call (avg over {repeat})")
    return out, dt

# ---------------------------
# 0) Warmup (optional but helpful)
# ---------------------------
try:
    r = requests.post(f"{BASE}/warmup", timeout=200)
    print("[warmup]", r.status_code, r.text[:80])
except Exception as e:
    print("[warmup] skipped:", e)

# ---------------------------
# 1) Health + Info
# ---------------------------
r = requests.get(f"{BASE}/health", timeout=200)
health = show(r, "HEALTH")
r = requests.get(f"{BASE}/info", timeout=200)
info = show(r, "INFO")
max_batch = (info or {}).get("max_batch", 256)
print("max_batch from server:", max_batch)

# ---------------------------
# 2) Dense (single) — backward compat via mode
# ---------------------------
payload = {"text": "canalización hormigonada 16 t polietileno", "mode": "dense"}
r = requests.post(f"{BASE}/encode", json=payload, timeout=200)
js = show(r, "ENCODE dense")
if js and "vec" in js:
    print("dense dim:", len(js["vec"]))
    print("dense L2≈1:", l2norm_ok(js["vec"]))

# Also test explicit flag (should be equivalent)
payload = {"text": "canalización hormigonada 16 t polietileno", "return_dense": True}
r = requests.post(f"{BASE}/encode", json=payload, timeout=200)
js2 = show(r, "ENCODE dense (flag)")
if js2 and "vec" in js2:
    print("dense dim (flag):", len(js2["vec"]))

# ---------------------------
# 3) Sparse (single) — backward compat via mode
# ---------------------------
payload = {"text": "canalización hormigonada 16 t polietileno", "mode": "sparse"}
r = requests.post(f"{BASE}/encode", json=payload, timeout=200)
js = show(r, "ENCODE sparse")
if js and "lexical_weights" in js:
    lw = js["lexical_weights"]
    samp = list(lw.items())[:5]
    print("sparse sample:", samp)
else:
    print("!! sparse missing 'lexical_weights' (check _decide_flags for mode=='sparse')")

# Also test explicit flag
payload = {"text": "canalización hormigonada 16 t polietileno", "return_sparse": True}
r = requests.post(f"{BASE}/encode", json=payload, timeout=200)
_ = show(r, "ENCODE sparse (flag)")

# ---------------------------
# 4) Dense (batch)
# ---------------------------
texts = [
    "canalización hormigonada 16 t polietileno",
    "entronque de tubos de 110 mm",
    "hormigón armado ha-25",
    "zanjas en terreno rocoso con reposición",
]
payload = {"texts": texts, "mode": "dense"}
r = requests.post(f"{BASE}/encode_batch", json=payload, timeout=200)
js = show(r, "ENCODE_BATCH dense")
if js and "vecs" in js:
    print("batch size:", len(js["vecs"]), "dim0:", len(js["vecs"][0]))
    print("dense[0] L2≈1:", l2norm_ok(js["vecs"][0]))

# ---------------------------
# 5) ColBERT token vectors
#    (A) Orthodox flags
# ---------------------------
payload = {
    "texts": texts,
    "return_colbert_vecs": True,
    "return_dense": False,
    "return_sparse": False,
    "max_length": 512,
}
r = requests.post(f"{BASE}/encode_batch", json=payload, timeout=200)
js = show(r, "ENCODE_BATCH colbert (flags)")
if js and "colbert_vecs" in js:
    first = js["colbert_vecs"][0]
    print("colbert first example:", len(first), "tokens ×", len(first[0]), "dim")
    print("colbert tokens L2≈1:", l2norm_tokens_ok(js["colbert_vecs"]))

#    (B) Convenience mode='colbert' (no flags)
payload = {"text": "prueba colbert", "mode": "colbert"}
r = requests.post(f"{BASE}/encode", json=payload, timeout=200)
js = show(r, "ENCODE colbert (mode)")
if js and "colbert_vecs" in js:
    M = js["colbert_vecs"]
    print("colbert(single):", len(M), "tokens ×", len(M[0]), "dim")
    print("colbert(single) tokens L2≈1:", l2norm_tokens_ok([M]))

# ---------------------------
# 6) Micro-benchmarks: latency & throughput
# ---------------------------
def bench_dense_single(n=32):
    q = {"text": "ejecución de canalización subterránea 220 kv", "mode": "dense"}
    _, dt = time_call(lambda: requests.post(f"{BASE}/encode", json=q, timeout=200), label="dense single", repeat=n)
    return dt

def bench_dense_batch(B=64, repeat=5):
    textsB = [f"texto de prueba {i}" for i in range(B)]
    q = {"texts": textsB, "mode": "dense"}
    _, dt = time_call(lambda: requests.post(f"{BASE}/encode_batch", json=q, timeout=200), label=f"dense batch B={B}", repeat=repeat)
    return dt, B

def bench_colbert_batch(B=32, L=256, repeat=3):
    textsB = [f"texto colbert {i}" for i in range(B)]
    q = {"texts": textsB, "mode": "colbert", "max_length": L}
    _, dt = time_call(lambda: requests.post(f"{BASE}/encode_batch", json=q, timeout=600), label=f"colbert batch B={B} L={L}", repeat=repeat)
    return dt, B, L

print("\n--- Benchmarks ---")
dt_single = bench_dense_single(n=8)
dt_B64, B = bench_dense_batch(B=min(64, max_batch), repeat=5)
dt_cB32, cB, L = bench_colbert_batch(B=min(32, max_batch), L=256, repeat=3)

# Rough throughput (vectors/sec). For colbert, this is *texts*/sec, not tokens/sec.
print(f"Throughput dense single: {1.0/dt_single:.1f} vec/s")
print(f"Throughput dense batch (B={B}): {B/dt_B64:.1f} vec/s")
print(f"Throughput colbert batch (B={cB}, L={L}): {cB/dt_cB32:.1f} texts/s")

# ---------------------------
# 7) Stress the max batch (bounded by server's max_batch)
# ---------------------------
B = max_batch
textsB = [f"texto stress {i}" for i in range(B)]
q = {"texts": textsB, "mode": "dense"}
r = requests.post(f"{BASE}/encode_batch", json=q, timeout=600)
js = show(r, f"STRESS dense B={B}")
if js and "vecs" in js:
    print("stress ok:", len(js["vecs"]) == B)

# ---------------------------
# 8) Error path sanity (too long text or too big batch)
# ---------------------------
too_big_batch = max_batch + 1
try:
    q = {"texts": ["x"] * too_big_batch, "mode": "dense"}
    r = requests.post(f"{BASE}/encode_batch", json=q, timeout=60)
    print("expected error status:", r.status_code, "| msg:", r.text[:200])
except Exception as e:
    print("error-path check:", e)

print("\nAll tests done.")


[warmup] 200 {"ok":true}
[HEALTH] 200 | keys: ['ok', 'device', 'cuda', 'model']
[INFO] 200 | keys: ['model', 'fp16', 'max_batch', 'max_text_len', 'defaults']
max_batch from server: 256
[ENCODE dense] 200 | keys: ['mode', 'vec']
dense dim: 1024
dense L2≈1: True
[ENCODE dense (flag)] 200 | keys: ['mode', 'vec']
dense dim (flag): 1024
[ENCODE sparse] 200 | keys: ['lexical_weights', 'mode']
sparse sample: [('26944', 0.176025390625), ('11313', 0.131103515625), ('3328', 0.09832763671875), ('266', 0.052215576171875), ('58108', 0.1270751953125)]
[ENCODE sparse (flag)] 200 | keys: ['lexical_weights', 'mode']
[ENCODE_BATCH dense] 200 | keys: ['vecs', 'mode']
batch size: 4 dim0: 1024
dense[0] L2≈1: True
[ENCODE_BATCH colbert (flags)] 200 | keys: ['colbert_vecs', 'mode']
colbert first example: 13 tokens × 1024 dim
colbert tokens L2≈1: True
[ENCODE colbert (mode)] 200 | keys: ['colbert_vecs', 'mode']
colbert(single): 4 tokens × 1024 dim
colbert(single) tokens L2≈1: True

--- Benchmarks ---
[time] d